In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2013_2014"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "OCQ_H.xpt"
csv_file = DATA_DIR / "OCQ_H_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Completely untouched pandas-decoded source.
#
source_df = df.copy()


print("=" * 90)
print("NHANES 2013-2014 OCQ_H XPT -> CSV")
print("=" * 90)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "OCD150",
    "OCQ180",
    "OCQ210",
    "OCD231",
    "OCD241",
    "OCQ260",
    "OCD270",
    "OCQ380",
    "OCD390G",
    "OCD391",
    "OCD392",
    "OCD395"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "OCQ_H column names/order differ from expected.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 6464:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 13:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 6,464 rows x 13 columns."
)


# ============================================================
# 4. DEFINE FIELD TYPES
# ============================================================

#
# After validated zero restoration, the attached OCQ_H
# contains NO genuine fractional values.
#
# Every field represents:
#
#     - respondent identifier
#     - questionnaire response code
#     - hours
#     - industry/occupation group code
#     - number of months
#
# Therefore all 13 fields can safely become nullable Int64.
#
#
# IMPORTANT:
#
# This applies only to OCQ_H.
#
# For Dietary, Laboratory, Examination, sample-weight,
# or other true continuous fields:
#
#     DO NOT round
#     DO NOT globally remove ".0"
#     DO NOT use float_format
#
# Genuine decimals remain exactly AS IS.
#

DECIMAL_COLUMNS = []


INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


if len(DECIMAL_COLUMNS) != 0:

    raise ValueError(
        "OCQ_H should contain no genuine "
        "decimal variables."
    )


if len(INTEGER_COLUMNS) != 13:

    raise ValueError(
        "Expected exactly 13 whole-number fields."
    )


print("\n--- FIELD TYPE PLAN ---")

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "Whole-number/code fields:",
    len(INTEGER_COLUMNS)
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT zero decoding artifact:
#
#     5.397605346934028e-79
#
# NEVER globally replace this without validating
# exact variable/count locations first.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 13 OCQ_H variables "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "Ordinary numeric zeros before correction:",
    f"{true_zero_total:,}"
)


print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)


print("\nTiny values by variable:")


print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Exact attached OCQ_H.xpt:
#
# OCD270 = 79
# OCD395 = 17
#
# TOTAL  = 96
#
#
# CDC:
#
# OCD270 = 0
#     Less than 1 month
#     Count = 79
#
# OCD395 = 0
#     Less than 1 month
#     Count = 17
#

EXPECTED_TINY_COUNTS = {
    "OCD270": 79,
    "OCD395": 17
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated OCQ_H attachment.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{actual_tiny_counts}"
    )


if tiny_total != 96:

    raise ValueError(
        f"Unexpected total tiny-value count: "
        f"{tiny_total:,}"
    )


if true_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary numeric zeros "
        "already existed before correction."
    )


print(
    "\nPASS: Exact 96-value tiny-zero "
    "pattern validated."
)


# ============================================================
# 7. RESTORE ONLY VALIDATED ZEROS
# ============================================================

#
# DO NOT:
#
#     df.replace(TINY_VALUE, 0)
#
# globally.
#

print("\n--- VALIDATED ZERO RESTORATION ---")


correction_total = 0


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected "
            f"{expected_count:,} tiny values, "
            f"found {actual_count:,}."
        )


    df.loc[
        mask,
        col
    ] = 0


    correction_total += actual_count


    print(
        f"{col}: "
        f"{actual_count:,} restored to 0"
    )


if correction_total != 96:

    raise ValueError(
        "Incorrect total zero-restoration count."
    )


print(
    "\nPASS:",
    f"{correction_total:,}",
    "validated zeros restored."
)


# ============================================================
# 8. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df contains exactly the values we expect
# after legitimate XPORT zero repair.
#
# No other values may change.
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 9. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny values remaining:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "after correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 10. CHECK FOR GENUINE FRACTIONAL VARIABLES
# ============================================================

#
# Run BEFORE converting anything to Int64.
#
# np.round() is used ONLY for detecting whether
# the values are fractional.
#
# It does NOT modify the dataframe.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal variables unexpectedly "
        "found in OCQ_H. They will NOT be rounded:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional variables."
)


# ============================================================
# 11. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 73557:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 83729:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains duplicate values."
    )


print(
    "PASS: SEQN 73557-83729 validated."
)


# ============================================================
# 12. CONVERT ALL PROVEN WHOLE-NUMBER FIELDS TO Int64
# ============================================================

#
# All 13 variables have now been proven whole-number.
#
# NO .round() is used during conversion.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():

            raise ValueError(
                f"{col} unexpectedly contains "
                "fractional values."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 13 OCQ_H fields converted "
    "to nullable Int64 without rounding."
)


# ============================================================
# 13. HELPER - EXACT FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    print(
        f"\n--- {column} ---"
    )


    print(
        "Observed:",
        actual_values
    )


    print(
        "Missing:",
        f"{actual_missing:,}"
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ from CDC.\n"
            f"Expected: {expected_values}\n"
            f"Actual:   {actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 14. HELPER - RANGE/SPECIAL CODE CHECK
# ============================================================

def check_range_special(
    column,
    low,
    high,
    expected_range_count,
    expected_missing,
    special_counts=None,
    expected_zero=None
):

    if special_counts is None:

        special_counts = {}


    s = df[column]


    range_count = int(
        s.between(
            low,
            high
        )
        .sum()
    )


    missing_count = int(
        s.isna()
        .sum()
    )


    zero_count = int(
        s.eq(0)
        .sum()
    )


    print(
        f"\n--- {column} ---"
    )


    print(
        f"Range {low}-{high}:",
        f"{range_count:,}"
    )


    print(
        "Zero:",
        f"{zero_count:,}"
    )


    print(
        "Missing:",
        f"{missing_count:,}"
    )


    if range_count != expected_range_count:

        raise ValueError(
            f"{column}: range count differs.\n"
            f"Expected: {expected_range_count:,}\n"
            f"Actual:   {range_count:,}"
        )


    if missing_count != expected_missing:

        raise ValueError(
            f"{column}: missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {missing_count:,}"
        )


    if expected_zero is not None:

        if zero_count != expected_zero:

            raise ValueError(
                f"{column}: zero count differs.\n"
                f"Expected: {expected_zero:,}\n"
                f"Actual:   {zero_count:,}"
            )


    for code, expected_count in special_counts.items():

        actual_count = int(
            s.eq(code)
            .sum()
        )


        print(
            f"Code {code}:",
            actual_count
        )


        if actual_count != expected_count:

            raise ValueError(
                f"{column}: code {code} count differs.\n"
                f"Expected: {expected_count:,}\n"
                f"Actual:   {actual_count:,}"
            )


    print(
        f"PASS: {column} range/special codes validated."
    )


# ============================================================
# 15. CDC CHECK - OCD150
# ============================================================

#
# Type of work done last week.
#
# CDC:
#
# 1 Working at job/business             3,345
# 2 Job/business but not at work           98
# 3 Looking for work                      200
# 4 Not working at job/business          2,816
# Missing                                    5
#

check_frequency(
    "OCD150",
    {
        1: 3345,
        2: 98,
        3: 200,
        4: 2816
    },
    5
)


# ============================================================
# 16. CDC CHECK - OCQ180
# ============================================================

#
# Hours worked last week.
#
# CDC released:
#
# 1-120 = 3,339
# 77777 = Refused      2
# 99999 = Don't know   4
# Missing = 3,119
#

check_range_special(
    "OCQ180",
    low=1,
    high=120,
    expected_range_count=3339,
    expected_missing=3119,
    special_counts={
        77777: 2,
        99999: 4
    },
    expected_zero=0
)


# ============================================================
# 17. OCD150 -> OCQ180 ROUTING
# ============================================================

#
# OCQ180 is populated exactly when:
#
#     OCD150 = 1
#
# Working at a job/business last week.
#

ocq180_expected = (
    df["OCD150"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


ocq180_present = (
    df["OCQ180"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


ocq180_route_mismatch = int(
    np.sum(
        ocq180_expected
        !=
        ocq180_present
    )
)


print(
    "\n--- OCD150 / OCQ180 ROUTING ---"
)


print(
    "Routing mismatches:",
    ocq180_route_mismatch
)


if ocq180_route_mismatch != 0:

    raise ValueError(
        "OCQ180 routing does not match OCD150=1."
    )


print(
    "PASS: OCQ180 routing validated."
)


# ============================================================
# 18. CDC CHECK - OCQ210
# ============================================================

#
# Usually works 35 or more hours/week.
#

check_frequency(
    "OCQ210",
    {
        1: 314,
        2: 693,
        7: 2,
        9: 2
    },
    5453
)


# ============================================================
# 19. OCQ210 ROUTING CHECK
# ============================================================

#
# OCQ210 applies to:
#
# OCD150 = 2
#
# OR
#
# OCD150 = 1 AND
# OCQ180 <= 34 / Refused / Don't know.
#

ocq210_expected = (

    df["OCD150"]
    .eq(2)

    |

    (
        df["OCD150"]
        .eq(1)

        &

        (
            df["OCQ180"]
            .le(34)

            |

            df["OCQ180"]
            .isin(
                [
                    77777,
                    99999
                ]
            )
        )
    )
)


ocq210_expected = (
    ocq210_expected
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


ocq210_present = (
    df["OCQ210"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


ocq210_route_mismatch = int(
    np.sum(
        ocq210_expected
        !=
        ocq210_present
    )
)


print(
    "\n--- OCQ210 ROUTING ---"
)


print(
    "Expected records:",
    int(
        ocq210_expected.sum()
    )
)


print(
    "Routing mismatches:",
    ocq210_route_mismatch
)


if ocq210_route_mismatch != 0:

    raise ValueError(
        "OCQ210 skip routing differs "
        "from expected questionnaire logic."
    )


print(
    "PASS: OCQ210 routing validated."
)


# ============================================================
# 20. CURRENT JOB INDUSTRY GROUP - OCD231
# ============================================================

#
# CDC industry group codes:
#
#     1-22
#
# Attached:
#
#     nonmissing = 3,418
#     missing    = 3,046
#

s = df["OCD231"]


ocd231_valid = int(
    s.between(
        1,
        22
    )
    .sum()
)


ocd231_missing = int(
    s.isna()
    .sum()
)


if not (
    ocd231_valid == 3418
    and
    ocd231_missing == 3046
):

    raise ValueError(
        "OCD231 range/count differs from CDC."
    )


print(
    "PASS: OCD231 industry-group codes "
    "1-22 validated."
)


# ============================================================
# 21. CURRENT JOB OCCUPATION GROUP - OCD241
# ============================================================

#
# CDC occupation group codes:
#
#     1-23
#
# Attached:
#
#     nonmissing = 3,421
#     missing    = 3,043
#

s = df["OCD241"]


ocd241_valid = int(
    s.between(
        1,
        23
    )
    .sum()
)


ocd241_missing = int(
    s.isna()
    .sum()
)


if not (
    ocd241_valid == 3421
    and
    ocd241_missing == 3043
):

    raise ValueError(
        "OCD241 range/count differs from CDC."
    )


print(
    "PASS: OCD241 occupation-group codes "
    "1-23 validated."
)


# ============================================================
# 22. CDC CHECK - OCQ260
# ============================================================

#
# Description of job/work situation.
#

check_frequency(
    "OCQ260",
    {
        1: 2846,
        2: 55,
        3: 122,
        4: 211,
        5: 199,
        6: 6,
        77: 4
    },
    3021
)


# ============================================================
# 23. CDC CHECK - OCD270
# ============================================================

#
# Number of months working in main job.
#
# CDC:
#
# 0 = Less than 1 month       79
# 1-720                     3,358
# 77777 Refused                 6
# 99999 Don't know              0
# Missing                   3,021
#
# The 79 legitimate zeros were decoded in pandas as
# TINY_VALUE and have now been restored.
#

check_range_special(
    "OCD270",
    low=1,
    high=720,
    expected_range_count=3358,
    expected_missing=3021,
    special_counts={
        77777: 6,
        99999: 0
    },
    expected_zero=79
)


# ============================================================
# 24. CURRENT-JOB ROUTING
# ============================================================

#
# OCQ260 and OCD270 are populated exactly when:
#
#     OCD150 in {1, 2}
#
# i.e. participant was working or had a job
# but was absent from work.
#

current_job_path = (
    df["OCD150"]
    .isin(
        [
            1,
            2
        ]
    )
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


for col in [
    "OCQ260",
    "OCD270"
]:

    present = (
        df[col]
        .notna()
        .to_numpy(
            dtype=bool
        )
    )


    mismatch = int(
        np.sum(
            current_job_path
            !=
            present
        )
    )


    if mismatch != 0:

        raise ValueError(
            f"{col} current-job routing mismatch: "
            f"{mismatch}"
        )


print(
    "PASS: OCQ260/OCD270 current-job "
    "routing validated."
)


# ============================================================
# 25. CDC CHECK - OCQ380
# ============================================================

#
# Main reason did not work last week.
#

check_frequency(
    "OCQ380",
    {
        1: 466,
        2: 449,
        3: 1113,
        4: 210,
        5: 84,
        6: 354,
        7: 139,
        77: 1
    },
    3648
)


# ============================================================
# 26. OCQ380 ROUTING CHECK
# ============================================================

#
# OCQ380 is populated exactly when:
#
#     OCD150 = 4
#
# Not working at a job/business.
#

ocq380_expected = (
    df["OCD150"]
    .eq(4)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


ocq380_present = (
    df["OCQ380"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


ocq380_mismatch = int(
    np.sum(
        ocq380_expected
        !=
        ocq380_present
    )
)


if ocq380_mismatch != 0:

    raise ValueError(
        "OCQ380 routing does not match OCD150=4."
    )


print(
    "PASS: OCQ380 routing validated."
)


# ============================================================
# 27. CDC CHECK - OCD390G
# ============================================================

#
# Longest job gate question.
#

check_frequency(
    "OCD390G",
    {
        1: 3397,
        2: 2442,
        3: 46,
        4: 563,
        7: 7,
        9: 4
    },
    5
)


# ============================================================
# 28. LONGEST JOB INDUSTRY GROUP - OCD391
# ============================================================

#
# CDC industry group codes:
#
#     1-22
#
# Nonmissing = 3,434
# Missing    = 3,030
#

s = df["OCD391"]


ocd391_valid = int(
    s.between(
        1,
        22
    )
    .sum()
)


ocd391_missing = int(
    s.isna()
    .sum()
)


if not (
    ocd391_valid == 3434
    and
    ocd391_missing == 3030
):

    raise ValueError(
        "OCD391 range/count differs from CDC."
    )


print(
    "PASS: OCD391 longest-job industry "
    "group codes validated."
)


# ============================================================
# 29. LONGEST JOB OCCUPATION GROUP - OCD392
# ============================================================

#
# CDC occupation group codes:
#
#     1-23
#
# Nonmissing = 3,437
# Missing    = 3,027
#

s = df["OCD392"]


ocd392_valid = int(
    s.between(
        1,
        23
    )
    .sum()
)


ocd392_missing = int(
    s.isna()
    .sum()
)


if not (
    ocd392_valid == 3437
    and
    ocd392_missing == 3027
):

    raise ValueError(
        "OCD392 range/count differs from CDC."
    )


print(
    "PASS: OCD392 longest-job occupation "
    "group codes validated."
)


# ============================================================
# 30. CDC CHECK - OCD395
# ============================================================

#
# Duration of longest job in months.
#
# CDC:
#
# 0 = Less than 1 month       17
# 1-720                     3,421
# 77777 Refused                 1
# 99999 Don't know              4
# Missing                   3,021
#
# The 17 zeros were decoded by pandas as TINY_VALUE.
#

check_range_special(
    "OCD395",
    low=1,
    high=720,
    expected_range_count=3421,
    expected_missing=3021,
    special_counts={
        77777: 1,
        99999: 4
    },
    expected_zero=17
)


# ============================================================
# 31. FINAL VALIDATED ZERO PATTERN
# ============================================================

#
# Original pandas-decoded XPT contained:
#
#     ordinary zeros = 0
#
# Therefore after repair the ONLY zeros should be:
#
# OCD270 = 79
# OCD395 = 17
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print(
    "\n--- FINAL ZERO PATTERN ---"
)


print(
    final_zero_counts
)


if final_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final zero pattern differs from "
        "validated OCQ_H pattern.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{final_zero_counts}"
    )


if sum(
    final_zero_counts.values()
) != 96:

    raise ValueError(
        "Unexpected final zero total."
    )


print(
    "PASS: Final zero pattern contains exactly "
    "79 OCD270 + 17 OCD395 zeros."
)


# ============================================================
# 32. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# The ONLY permitted numerical changes:
#
# OCD270:
#     79 TINY_VALUE -> 0
#
# OCD395:
#     17 TINY_VALUE -> 0
#
# Everything else must remain numerically identical.
#

print(
    "\n--- VALUE FIDELITY BEFORE EXPORT ---"
)


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 13 variables match the exact "
    "expected corrected XPT values."
)


# ============================================================
# 33. EXPORT CSV
# ============================================================

#
# Standard NHANES fidelity export.
#
# NO:
#
#     round()
#
# NO:
#
#     float_format=
#
# OCQ_H itself contains no genuine decimals.
#
# This export pattern remains consistent with your
# Dietary/Lab/Exam files where genuine decimal values
# must remain AS IS.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)


print(
    csv_file
)


# ============================================================
# 34. RE-READ WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 35. EXACT CORRECTED XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT CORRECTED XPT -> CSV FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV numeric differences:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 13 CSV variables round-trip "
    "to exact corrected XPT numeric values."
)


# ============================================================
# 36. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print(
    "\n--- MISSING VALUE FIDELITY ---"
)


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 37. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 38. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values.eq(TINY_VALUE)
        .sum()
    )


print(
    "\n--- FINAL TINY VALUE CHECK ---"
)


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 39. RAW CSV .0 CHECK
# ============================================================

#
# OCQ_H contains NO genuine decimal fields.
#
# Therefore a .0 check across all 13 variables is safe
# for this specific file.
#
# Examples:
#
# SEQN:
#     73557.0 -> 73557
#
# OCQ180:
#     40.0 -> 40
#
# OCD270:
#     tiny -> 0
#     120.0 -> 120
#
# OCD395:
#     tiny -> 0
#
#
# DO NOT use this global check on mixed decimal datasets.
#

print(
    "\n--- RAW CSV .0 CHECK ---"
)


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting found:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary '.0' "
    "in any OCQ_H field."
)


# ============================================================
# 40. FINAL CSV ZERO-PATTERN CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values.eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print(
    "\n--- FINAL CSV ZERO PATTERN ---"
)


print(
    csv_zero_counts
)


if csv_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final CSV zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{csv_zero_counts}"
    )


print(
    "PASS: Final CSV zero pattern exactly matches "
    "the 96 validated restorations."
)


# ============================================================
# 41. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    tokens = raw_csv_df[col]


    scientific_count += int(
        tokens.str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print(
    "\n--- SCIENTIFIC NOTATION CHECK ---"
)


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 42. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 73557:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 83729:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn.duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final CSV SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 43. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 90
)


print(
    "OCQ_H CONVERSION COMPLETE"
)


print(
    "=" * 90
)


print(
    "Cycle: 2013-2014"
)


print(
    "Component: Questionnaire - Occupation"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "Variables containing tiny artifacts:",
    len(EXPECTED_TINY_COUNTS)
)


print(
    "OCD270 zero restorations:",
    EXPECTED_TINY_COUNTS["OCD270"]
)


print(
    "OCD395 zero restorations:",
    EXPECTED_TINY_COUNTS["OCD395"]
)


print(
    "Total validated zero restorations:",
    correction_total
)


print(
    "Genuine decimal variables: NONE"
)


print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)


print(
    "Currently working:",
    int(
        df["OCD150"]
        .eq(1)
        .sum()
    )
)


print(
    "Job but not at work:",
    int(
        df["OCD150"]
        .eq(2)
        .sum()
    )
)


print(
    "Looking for work:",
    int(
        df["OCD150"]
        .eq(3)
        .sum()
    )
)


print(
    "Not working:",
    int(
        df["OCD150"]
        .eq(4)
        .sum()
    )
)


print(
    "OCD270 <1 month:",
    int(
        df["OCD270"]
        .eq(0)
        .sum()
    )
)


print(
    "OCD395 <1 month:",
    int(
        df["OCD395"]
        .eq(0)
        .sum()
    )
)


print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)


print(
    "Missing-position differences: "
    "0 expected"
)


print(
    "Tiny values remaining: "
    "0 expected"
)


print(
    "Unnecessary '.0': "
    "0 expected"
)


print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2013-2014 OCQ_H XPT -> CSV
Rows: 6,464
Columns: 13
PASS: Structure = 6,464 rows x 13 columns.

--- FIELD TYPE PLAN ---
Genuine decimal fields: []
Whole-number/code fields: 13

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before correction: 0
Tiny XPORT artifacts: 96

Tiny values by variable:
OCD270    79
OCD395    17

PASS: Exact 96-value tiny-zero pattern validated.

--- VALIDATED ZERO RESTORATION ---
OCD270: 79 restored to 0
OCD395: 17 restored to 0

PASS: 96 validated zeros restored.

Tiny values remaining: 0
PASS: No tiny XPORT artifacts remain.

--- FRACTIONAL VALUE CHECK ---
SEQN: 0 fractional observations
OCD150: 0 fractional observations
OCQ180: 0 fractional observations
OCQ210: 0 fractional observations
OCD231: 0 fractional observations
OCD241: 0 fractional observations
OCQ260: 0 fractional observations
OCD270: 0 fractional observations
OCQ380: 0 fractional observations
OCD390G: 0 fractional observations
OCD391: 0 fractional observations
OCD392: 0 fractional o